In [ ]:
import pandas as pd
import os

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
import torch
torch.manual_seed(100)

import random
random.seed(15)

import numpy as np
np.random.seed(30)

In [ ]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
DIR_INPUTS = './data_raw/'
DIR_RUNTIME_DATA = './data_runtime/'
DIR_RUNTIME_RESULTS = './results_runtime/'
TUNE = False
TRANS_ONLY=False

### Create train-validation and test sets (into csvs)

In [ ]:
MAX_STATES = 5
N_TIME = 11

In [ ]:
MAX_STATES

## Load train val and test sets

In [ ]:
from monotonic_nn_surv_surf.utils.datasets_def import DatasetFeatANDsurf
from torch.utils.data import DataLoader

In [ ]:
ds_train = DatasetFeatANDsurf(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_train.csv'),
    path_surf=os.path.join(DIR_RUNTIME_DATA,'df_surfs_train.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
)
loader_train = DataLoader(ds_train, batch_size=1000,shuffle=False)

ds_val = DatasetFeatANDsurf(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_val.csv'),
    path_surf=os.path.join(DIR_RUNTIME_DATA,'df_surfs_val.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
)
loader_val = DataLoader(ds_val, batch_size=1000)

ds_test = DatasetFeatANDsurf(
    path_feat_by_subj=os.path.join(DIR_RUNTIME_DATA,'df_features_test.csv'),
    path_surf=os.path.join(DIR_RUNTIME_DATA,'df_surfs_test.csv'),
    max_grade=MAX_STATES, 
    max_time=N_TIME,
)
loader_test = DataLoader(ds_test, batch_size=1000)

In [ ]:
len(ds_train)

In [ ]:
len(ds_val)

## Specify model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Load model

In [ ]:
dir_logs = os.path.join(DIR_RUNTIME_RESULTS, 'lightning_logs')
dir_logs

In [ ]:
latest_ver = sorted(os.listdir(dir_logs))[-1]
latest_ver

In [ ]:
checkpoint_path = [i for i in os.listdir(os.path.join(dir_logs, f'{latest_ver}/checkpoints')) if i.endswith('ckpt')][-1]
checkpoint_path = os.path.join(dir_logs, f'{latest_ver}/checkpoints/{checkpoint_path}')
checkpoint_path

In [ ]:
import json
 
if TUNE:
   dir_checkpoint = DIR_RUNTIME_RESULTS
else:
    dir_checkpoint = DIR_RUNTIME_RESULTS


# Opening JSON file
with open(os.path.join(dir_checkpoint,"best_hyper_params.json"), 'r') as openfile:
 
    # Reading from json file
    best_hyper_params = json.load(openfile)
best_hyper_params

In [ ]:
from monotonic_nn_surv_surf.utils.surv_surf_latent import SurvSurfLatent, LatentFeatFC
from monotonic_nn_surv_surf.utils.pl_model_wrapper import LitSurvSurfReg

In [ ]:
n_feat_neurons = best_hyper_params['n_feat_neurons']
n_monoton_neurons = best_hyper_params['n_monoton_neurons']
n_monotone_layers = best_hyper_params['n_monotone_layers']
n_feat_layers = best_hyper_params['n_feat_layers']
p_dropout = best_hyper_params['p_dropout']
learning_rate = best_hyper_params['learning_rate']

model = SurvSurfLatent(
    mono_net_sizes=[n_feat_neurons] + [n_monoton_neurons]*n_monotone_layers + [1],
    latent_feat_transformer=LatentFeatFC(
        input_size=3, 
        output_size=n_feat_neurons, 
        neurons_per_layer=(n_feat_layers-1)*[n_feat_neurons],
        dropout_p=p_dropout
    ),
)

model_lit = LitSurvSurfReg(model=model, lr=learning_rate, print_epoch=True)

model_lit_loaded = model_lit.load_from_checkpoint(checkpoint_path)
model_loaded_core = model_lit_loaded.model
model_loaded_core = model_loaded_core.to(device)
model_loaded_core.eval()

## Train-set performance

In [ ]:
pred = []
truth = []
ts_all = []
gs_all = []
with torch.no_grad():
    for i, tdata in enumerate(loader_train):
        xs, ts, gs, ys, weights = tdata
        xs, ts, gs, ys = xs.to(device), ts.to(device), gs.to(device), ys.to(device)
        toutputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
        ts_all += list(ts.cpu().numpy()[:,0])
        gs_all += list(gs.cpu().numpy()[:,0])
        truth += list(ys.cpu().numpy()[:,0])
        pred += list(toutputs.cpu().numpy()[:,0])

pred = np.array(pred)
truth = np.array(truth)
ts_all = np.array(ts_all)
gs_all = np.array(gs_all)


df_train_results = ds_train.observed.copy()
df_train_results['pred'] = pred
df_train_results['truth'] = truth
df_train_results['ts'] = (ts_all*N_TIME).round(0)
df_train_results['gs'] = (gs_all*MAX_STATES).round(0)
df_train_results.head()

del pred
del truth
del ts_all
del gs_all

In [ ]:
df_train_results.head()

In [ ]:
((df_train_results['truth'] - df_train_results['pred'])**2).mean()

In [ ]:
fig, ax = plt.subplots(1,1)

sns.scatterplot(
    data=df_train_results,
    x='t',
    y=(df_train_results['truth'] - df_train_results['pred']),
    hue='g',
    alpha=0.5,
    s=5,
)

In [ ]:
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)

fig, ax = plt.subplots(1,1, figsize=(16, 4))

# Change major ticks to show every 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Change minor ticks to show every 0.02
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.grid(which='major', axis='y')
ax.grid(which='minor', axis='y', ls=':')

sns.boxplot(
    data=df_train_results,
    x='t',
    y=(df_train_results['pred'] - df_train_results['truth']),
    hue='g',
    boxprops={'alpha':0.5}
)
ax.set(ylabel='prediction - truth')


In [ ]:
subj_by_max_err = df_train_results.groupby('subject').apply(lambda x: (x['truth'] - x['pred']).abs().max())

sns.ecdfplot(subj_by_max_err)

In [ ]:
df_train_results.head()

In [ ]:
suf_mse = df_train_results.groupby(['t','g']).apply(
    lambda df: np.mean((df['truth'] - df['pred'])**2)
).reset_index()
suf_mse.columns

In [ ]:
suf_mse = df_train_results.groupby(['t','g']).apply(
    lambda df: np.mean((df['truth'] - df['pred'])**2)
).reset_index().pivot(index='g', columns='t', values=0)

fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=suf_mse,
    ax=ax,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)

In [ ]:
suf_var = df_train_results.groupby(['t','g'])['truth'].std().reset_index().pivot(index='g', columns='t', values='truth')**2

fig, ax = plt.subplots(1,1, figsize=(10, 2))
sns.heatmap(
    data=suf_var,
    ax=ax,
    # vmin=0,
    # vmax=1,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)
ax.set(title=f'truth for subj')

In [ ]:
suf_var = df_train_results.groupby(['t','g'])['truth'].std().reset_index().pivot(index='g', columns='t', values='truth')**2

suf_plot = suf_mse/suf_var
fig, ax = plt.subplots(1,1, figsize=(10, 2))
sns.heatmap(
    data=suf_plot,
    ax=ax,
    # vmin=0,
    # vmax=1,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)
ax.set(title=f'truth for subj')

In [ ]:
fig, ax = plt.subplots(1,1)

sns.scatterplot(
    data=df_train_results,
    x='truth',
    y='pred',
    hue='t',
    alpha=0.5,
    s=5,
)


## Test set performance

In [ ]:
pred = []
truth = []
ts_all = []
gs_all = []
with torch.no_grad():
    for i, tdata in enumerate(loader_test):
        xs, ts, gs, ys, weights = tdata
        xs, ts, gs, ys = xs.to(device), ts.to(device), gs.to(device), ys.to(device)
        toutputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
        ts_all += list(ts.cpu().numpy()[:,0])
        gs_all += list(gs.cpu().numpy()[:,0])
        truth += list(ys.cpu().numpy()[:,0])
        pred += list(toutputs.cpu().numpy()[:,0])

pred = np.array(pred)
truth = np.array(truth)
ts_all = np.array(ts_all)
gs_all = np.array(gs_all)


df_test_results = ds_test.observed.copy()
df_test_results['pred'] = pred
df_test_results['truth'] = truth
df_test_results['ts'] = (ts_all*N_TIME).round(0)
df_test_results['gs'] = (gs_all*MAX_STATES).round(0)
df_test_results.head()

del pred
del truth
del ts_all
del gs_all

In [ ]:
df_test_results.head()

In [ ]:
((df_test_results['truth'] - df_test_results['pred'])**2).mean()

In [ ]:
fig, ax = plt.subplots(1,1)

sns.scatterplot(
    data=df_test_results,
    x='t',
    y=(df_test_results['truth'] - df_test_results['pred']),
    hue='g',
    alpha=0.5,
    s=5,
)

In [ ]:
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)

fig, ax = plt.subplots(1,1, figsize=(16, 4))

# Change major ticks to show every 0.1.
ax.yaxis.set_major_locator(MultipleLocator(0.1))

# Change minor ticks to show every 0.02
ax.yaxis.set_minor_locator(AutoMinorLocator(5))
ax.grid(which='major', axis='y')
ax.grid(which='minor', axis='y', ls=':')

sns.boxplot(
    data=df_test_results,
    x='t',
    y=(df_test_results['pred'] - df_test_results['truth']),
    hue='g',
    boxprops={'alpha':0.5}
)



In [ ]:
fig, ax = plt.subplots(1,1)
subj_by_max_err = df_train_results.groupby('subject').apply(lambda x: (x['truth'] - x['pred']).abs().max())
sns.ecdfplot(subj_by_max_err, ax=ax, label='training-set')

subj_by_max_err = df_test_results.groupby('subject').apply(lambda x: (x['truth'] - x['pred']).abs().max())
sns.ecdfplot(subj_by_max_err, ax=ax, label='test-set')

ax.legend()

In [ ]:
df_test_results.head()

In [ ]:
suf_mse = df_test_results.groupby(['t','g']).apply(
    lambda df: np.mean((df['truth'] - df['pred'])**2)
).reset_index()
suf_mse.columns

In [ ]:
suf_mse = df_test_results.groupby(['t','g']).apply(
    lambda df: np.mean((df['truth'] - df['pred'])**2)
).reset_index().pivot(index='g', columns='t', values=0)

fig, ax = plt.subplots(1,1, figsize=(15, 3))
sns.heatmap(
    data=suf_mse,
    ax=ax,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)

In [ ]:
suf_var = df_test_results.groupby(['t','g'])['truth'].std().reset_index().pivot(index='g', columns='t', values='truth')**2

fig, ax = plt.subplots(1,1, figsize=(10, 2))
sns.heatmap(
    data=suf_var,
    ax=ax,
    # vmin=0,
    # vmax=1,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)
ax.set(title=f'truth for subj')

In [ ]:
suf_var = df_test_results.groupby(['t','g'])['truth'].std().reset_index().pivot(index='g', columns='t', values='truth')**2

suf_plot = suf_mse/suf_var
fig, ax = plt.subplots(1,1, figsize=(10, 2))
sns.heatmap(
    data=suf_plot,
    ax=ax,
    # vmin=0,
    # vmax=1,
    cbar_kws={'label': 'P(reaching max grade by time)'},
    annot=True
)
ax.set(title=f'truth for subj')

In [ ]:
fig, ax = plt.subplots(1,1)

sns.scatterplot(
    data=df_test_results,
    x='truth',
    y='pred',
    hue='t',
    style='g',
    alpha=0.5,
    s=10,
)


## Plot example surface

In [ ]:
def plot_surfs_for_subj(sbj, df_X_y_all, figsize_single=(6,4)):
    df_X_y = df_X_y_all.loc[
        df_X_y_all['subject'] == sbj,:
    ]

    # compute the surface (smooth)
    xs = df_X_y.iloc[[0]][['0', '1', '2']].values
    max_time = N_TIME-1
    t = np.arange(0, max_time, max_time/50)
    g = np.arange(0.01, MAX_STATES, MAX_STATES/50)

    ts, gs = np.meshgrid(t, g)
    ts = ts.flatten()[:,None].astype(np.float32)/ds_test.max_time
    gs = gs.flatten()[:,None].astype(np.float32)/ds_test.max_grade
    xs = np.repeat(xs, ts.shape[0], axis=0).astype(np.float32)
    xs = torch.from_numpy(xs)
    ts = torch.from_numpy(ts)
    gs = torch.from_numpy(gs)
    xs, ts, gs = xs.to(device), ts.to(device), gs.to(device)
    outputs = model_loaded_core(ts=ts, gs=gs, xs=xs)
    pred = outputs.detach().cpu().numpy()[:,0]


    df_surf_long = pd.DataFrame()
    df_surf_long['t'] = ts.detach().cpu().numpy()[:,0]*ds_test.max_time
    df_surf_long['g'] = gs.detach().cpu().numpy()[:,0]*ds_test.max_grade
    df_surf_long['pred'] = pred


    # get the discrete versions of the surface (true and pred)
    df_true_surf = df_X_y.pivot(index='g', columns='t', values='y').sort_index(ascending=False)
    df_pred_surf_discre = df_X_y.pivot(index='g', columns='t', values='pred').sort_index(ascending=False)

    # plot the surfaces
    df_surf = df_surf_long.pivot(index='g', columns='t', values='pred')
    T = np.repeat(df_surf.columns.values[None,:], df_surf.index.size, axis=0)
    G = np.repeat(df_surf.index.values[:,None], df_surf.columns.size, axis=1)

    
    fig = plt.figure(figsize=figsize_single)
    # 3D plot
    ax = plt.axes(projection='3d')
    T, G = np.meshgrid(t, g)
    Y = df_surf.values

    surf = ax.plot_surface(T,G, Y, cmap = plt.cm.cividis)

    # Set axes label
    ax.set_xlabel('t', labelpad=20)
    ax.set_ylabel('g', labelpad=20)
    ax.set_zlabel('y', labelpad=20)
    # ax.set_zlim(-0.01, 1.01)
    ax.view_init(30, 210)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(1,1, figsize=figsize_single)
    # contour plot
    cp = ax.contourf(T,G,Y)
    fig.colorbar(cp)
    ax.set_xlabel('t')
    ax.set_ylabel('g')
    plt.show()

    plt.figure(figsize=figsize_single)
    sns.heatmap(df_true_surf, vmin=0, vmax=1, cmap='viridis')
    plt.show()

    plt.figure(figsize=figsize_single)
    sns.heatmap(df_pred_surf_discre, vmin=0, vmax=1, cmap='viridis')
    plt.show()

In [ ]:
plot_surfs_for_subj(sbj=1065, df_X_y_all=df_test_results, figsize_single=(6,4))



In [ ]:
plot_surfs_for_subj(sbj=1201, df_X_y_all=df_test_results, figsize_single=(6,4))


In [ ]:
plot_surfs_for_subj(sbj=1525, df_X_y_all=df_test_results, figsize_single=(6,4))
